# Devstral and Voxtral on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Mistral's specialist models: Devstral 2 for agentic coding, and the Voxtral models,
which take audio as well as text. The general-purpose Mistral models and the size ladder
are in [`01-mistral-text-and-sizes`](01-mistral-text-and-sizes.ipynb).

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| Devstral 2 123B | `mistral.devstral-2-123b` | `mistral.devstral-2-123b` | Agentic coding |
| Voxtral Small 24B | `mistral.voxtral-small-24b-2507` | `mistral.voxtral-small-24b-2507` | Speech and text; the transcription model here |
| Voxtral Mini 3B | `mistral.voxtral-mini-3b-2507` | `mistral.voxtral-mini-3b-2507` | Smaller and faster |

IDs as of September 2026, in `us-east-1`. Which Regions carry each model changes as
models launch, so the setup cell checks your Region.

On `bedrock-mantle` these models serve the OpenAI-compatible **Chat Completions** API at
the bare `/v1` path, which this notebook uses. On `bedrock-runtime` they serve
**Converse** under the same IDs.

**Covered:** code generation, checking generated code without running it, a coding agent
with tools, a structured code review, and transcription.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK. From `_shared/bedrock.py`,
this repository's own helper module, it takes an availability check, a lenient JSON
parser, static checks for generated code, and the sample audio clip.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import ast
import json
import sys

from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, parse_json_lenient

REGION = "us-east-1"
DEVSTRAL = "mistral.devstral-2-123b"
VOXTRAL_SMALL = "mistral.voxtral-small-24b-2507"
VOXTRAL_MINI = "mistral.voxtral-mini-3b-2507"

# A short-term Bedrock API key minted from your AWS credentials. Mint it when you
# build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=180,
)

for model in (DEVSTRAL, VOXTRAL_SMALL, VOXTRAL_MINI):
    served = [name for name, ok in endpoints_for(model, REGION).items() if ok]
    print(f"{model:32} served by: {', '.join(served) or 'neither endpoint in ' + REGION}")

mistral.devstral-2-123b          served by: mantle, runtime
mistral.voxtral-small-24b-2507   served by: mantle, runtime
mistral.voxtral-mini-3b-2507     served by: mantle, runtime


## 1. Code generation with Devstral

Ask for a function to a specification. A low `temperature` keeps the output close to the
model's most likely answer, which suits code; this family accepts `temperature` and
`top_p`.

In [2]:
SPEC = """Write a Python function `retry(fn, attempts=3, base_delay=0.1)` that:
- calls fn() and returns its result
- retries on any exception, up to `attempts` calls in total
- sleeps base_delay * (2 ** attempt) between attempts
- re-raises the last exception if every attempt fails
Include a short docstring. Reply with one Python code block and nothing else."""

response = client.chat.completions.create(
    model=DEVSTRAL,
    messages=[{"role": "user", "content": SPEC}],
    max_tokens=1000,
    temperature=0.2,
)
generated = response.choices[0].message.content or ""
print(generated.strip()[:1200])
print(f"\nfinish_reason={response.choices[0].finish_reason}")

```python
import time

def retry(fn, attempts=3, base_delay=0.1):
    """
    Retries calling `fn()` up to `attempts` times with exponential backoff.

    Args:
        fn: The function to call.
        attempts: Maximum number of attempts (default: 3).
        base_delay: Initial delay in seconds (default: 0.1).

    Returns:
        The result of `fn()` if successful.

    Raises:
        The last exception if all attempts fail.
    """
    last_exception = None
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as e:
            last_exception = e
            if attempt < attempts - 1:
                time.sleep(base_delay * (2 ** attempt))
    raise last_exception
```

finish_reason=stop


## 2. Checking the code without running it

Model output is untrusted input, and this kernel holds your AWS credentials, so do not
`exec()` it here. `inspect_code` and `check_spec` from `_shared/bedrock.py` parse the
source with `ast` and report what it declares, without running anything. The
specification asks for a backoff, so the cell also checks that the code calls `sleep`.
To run generated code, use a sandbox with no credentials and no network access, such as
Amazon Bedrock AgentCore Code Interpreter.

In [3]:
from bedrock import check_spec, extract_code_block, inspect_code

source = extract_code_block(generated)
info = inspect_code(source)
verdict = check_spec(source, function="retry", params=["fn", "attempts", "base_delay"])
sleeps = any(name.endswith("sleep") for name in info["calls"])

print("parses   :", info["parses"], info["error"] or "")
print("functions:", info["functions"])
print("calls    :", sorted(set(info["calls"])))
print(f"\nsignature matches: {verdict['ok']}   backs off with sleep: {sleeps}")
print("meets the spec" if verdict["ok"] and sleeps else f"does not meet the spec: {verdict['reason'] or 'no sleep call'}")

parses   : True 
functions: {'retry': ['fn', 'attempts', 'base_delay']}
calls    : ['fn', 'range', 'sleep']

signature matches: True   backs off with sleep: True
meets the spec


## 3. A coding agent with tools

Devstral is built for working through a codebase with tools. The workspace here is a
dictionary with two files, and the task is to make `fetch()` honour a retry setting.
`check_retries` checks the result with `ast`, so the model's code is never run. The loop
validates each call against the tool's parameters and stops after eight rounds.

In [4]:
WORKSPACE = {
    "config.py": "TIMEOUT = 30\nRETRIES = 3\n",
    "client.py": ("import config\n\n\ndef fetch(url):\n    # TODO: honour config.RETRIES\n"
                  "    return _get(url, timeout=config.TIMEOUT)\n\n\n"
                  "def _get(url, timeout):\n    return {'url': url, 'timeout': timeout}\n"),
}


def list_files() -> dict:
    return {"files": sorted(WORKSPACE)}


def read_file(path: str) -> dict:
    return {"path": path, "content": WORKSPACE.get(path), "exists": path in WORKSPACE}


def write_file(path: str, content: str) -> dict:
    WORKSPACE[path] = content
    return {"path": path, "bytes": len(content)}


def check_retries(path: str) -> dict:
    """Pass when fetch() loops and reads config.RETRIES, checked with ast. Never runs it."""
    try:
        tree = ast.parse(WORKSPACE.get(path, ""))
    except SyntaxError as exc:
        return {"ok": False, "reason": f"syntax error: {exc.msg}"}
    fetch = next((n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "fetch"), None)
    if fetch is None:
        return {"ok": False, "reason": "no fetch() in " + path}
    loops = any(isinstance(n, (ast.For, ast.While)) for n in ast.walk(fetch))
    reads = any(isinstance(n, ast.Attribute) and n.attr == "RETRIES" for n in ast.walk(fetch))
    return {"ok": loops and reads, "loops": loops, "reads_RETRIES": reads}

In [5]:
def tool(name: str, description: str, *params: str) -> dict:
    schema = {"type": "object", "properties": {p: {"type": "string"} for p in params}, "required": list(params)}
    return {"type": "function", "function": {"name": name, "description": description, "parameters": schema}}


CODE_TOOLS = [
    tool("list_files", "List the files in the workspace."),
    tool("read_file", "Read a file from the workspace.", "path"),
    tool("write_file", "Write a file to the workspace.", "path", "content"),
    tool("check_retries", "Check whether fetch() in a file retries.", "path"),
]
DISPATCH = {"list_files": list_files, "read_file": read_file, "write_file": write_file,
            "check_retries": check_retries}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t["function"] for t in CODE_TOOLS if t["function"]["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return DISPATCH[name](**args)

In [6]:
TASK = ("client.py has a TODO: fetch() ignores config.RETRIES. Rewrite fetch() so it retries up "
        "to config.RETRIES times, write the file back, then run check_retries on it.")

messages = [{"role": "user", "content": TASK}]
for round_no in range(1, 9):  # bounded, so an agent that cannot finish stops
    response = client.chat.completions.create(
        model=DEVSTRAL, messages=messages, tools=CODE_TOOLS, max_tokens=2000, temperature=0.2
    )
    message = response.choices[0].message
    if not message.tool_calls:
        break
    messages.append(message.model_dump(exclude_none=True))
    for call in message.tool_calls:
        args = parse_json_lenient(call.function.arguments or "{}")
        result = run_tool(call.function.name, args)
        print(f"round {round_no}: {call.function.name}({', '.join(args)}) -> {json.dumps(result)[:80]}")
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
else:
    raise RuntimeError("the agent did not finish in 8 rounds")

print("\nanswer:", (message.content or "").strip()[:300])
print("\nclient.py now:\n" + WORKSPACE["client.py"])
print("check:", check_retries("client.py"))

round 1: read_file(path) -> {"path": "client.py", "content": "import config\n\n\ndef fetch(url):\n    # TODO


round 2: write_file(path, content) -> {"path": "client.py", "bytes": 378}


round 3: check_retries(path) -> {"ok": true, "loops": true, "reads_RETRIES": true}



answer: The `fetch()` function has been successfully updated to retry up to `config.RETRIES` times, and the `check_retries` test confirms that it works as expected.

client.py now:
import config
import time


def fetch(url):
    retries = config.RETRIES
    for attempt in range(retries):
        try:
            return _get(url, timeout=config.TIMEOUT)
        except Exception:
            if attempt < retries - 1:
                time.sleep(1)
            else:
                raise


def _get(url, timeout):
    return {'url': url, 'timeout': timeout}

check: {'ok': True, 'loops': True, 'reads_RETRIES': True}


## 4. A structured code review

Ask for review findings as JSON, so a pipeline can act on them: where each problem is,
what it is, and how severe. The code builds an SQL query from user input, so the cell
checks that the review flagged SQL injection in `build_query`.

In [7]:
SUSPECT = '''
import os


def load_secret(name):
    return os.environ[name]


def build_query(table, user_input):
    return f"SELECT * FROM {table} WHERE name = '{user_input}'"


def divide_all(values, divisor):
    return [v / divisor for v in values]
'''
FINDING = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "issue": {"type": "string"},
        "severity": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["location", "issue", "severity"],
    "additionalProperties": False,
}
REVIEW_SCHEMA = {
    "type": "object",
    "properties": {"findings": {"type": "array", "items": FINDING},
                   "overall": {"type": "string", "enum": ["approve", "request_changes"]}},
    "required": ["findings", "overall"],
    "additionalProperties": False,
}

In [8]:
response = client.chat.completions.create(
    model=DEVSTRAL,
    messages=[{"role": "user", "content": f"Review this code for bugs and security issues.\n```python\n{SUSPECT}```"}],
    response_format={"type": "json_schema", "json_schema": {"name": "review", "strict": True, "schema": REVIEW_SCHEMA}},
    max_tokens=2000,
    temperature=0.2,
)
choice = response.choices[0]
if choice.finish_reason != "stop":
    raise RuntimeError(f"incomplete output (finish_reason={choice.finish_reason}); raise max_tokens")

review = parse_json_lenient(choice.message.content)
print("overall:", review["overall"])
for finding in review["findings"]:
    print(f"[{finding['severity']:6}] {finding['location']}: {finding['issue'][:80]}")
flagged = any("build_query" in f["location"] and "sql" in f["issue"].lower() for f in review["findings"])
print(f"\nflagged SQL injection in build_query: {flagged}")

overall: request_changes
[medium] load_secret: Missing error handling for missing environment variables
[high  ] build_query: SQL injection vulnerability due to direct string interpolation
[medium] divide_all: No validation for divisor being zero

flagged SQL injection in build_query: True


## 5. Transcription with Voxtral

Send audio as an `input_audio` content part holding base64 data and its `format`, next
to a text part with the instruction. An `audio_url` part returns 400 `Failed to
deserialize the JSON body into the target type`. The clip is seven seconds of speech
from a public AWS talk and ships with this repository, so the transcript is known; the
cell scores keywords, because speech recognition differs on filler words and word
boundaries. In September 2026 runs both Voxtral Small and Voxtral Mini ended the
transcript around "In Amazon S3, you" with `finish_reason` `stop`, well inside
`max_tokens`, and left out the clip's last clause. Voxtral also answers text-only
requests, like any chat model.

In [9]:
from bedrock import CLIP_KEYWORDS, CLIP_TRANSCRIPT, clip_mp3_b64, keyword_recall

response = client.chat.completions.create(
    model=VOXTRAL_SMALL,
    messages=[{"role": "user", "content": [
        {"type": "input_audio", "input_audio": {"data": clip_mp3_b64(), "format": "mp3"}},
        {"type": "text", "text": "Transcribe this audio verbatim."},
    ]}],
    max_tokens=300,
)
heard = (response.choices[0].message.content or "").strip()
hits, total = keyword_recall(heard, CLIP_KEYWORDS)
print("heard    :", heard)
print("reference:", CLIP_TRANSCRIPT)
print(f"\nfinish_reason: {response.choices[0].finish_reason}   keywords found: {hits}/{total}")

heard    : You can save costs by leveraging the tiering mechanism in Amazon S3. In Amazon S3, you
reference: you can save cost by leveraging the tiering mechanism in Amazon S3, in Amazon S3 you can store your infrequently accessed data

finish_reason: stop   keywords found: 3/3


## Things that differ

What trips people up on these models, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Responses API | 400 `The model 'mistral.voxtral-small-24b-2507' does not support the '/v1/responses' API`, although the Voxtral model cards list Responses | Use Chat Completions. |
| Path prefix | On `bedrock-mantle`, `/openai/v1/chat/completions` returns 400 ``model `…` isn't supported on this route``. On `bedrock-runtime` that path answers 200 for `mistral.devstral-2-123b` and `mistral.voxtral-small-24b-2507` under their bare IDs (October 2026) | Use the bare `/v1` path on `bedrock-mantle`. The prefix is not interchangeable between the endpoints: `bedrock-runtime` has an `/openai/v1` route of its own that serves Devstral and Voxtral, so the mantle 400 describes that route rather than the models. |
| Empty answer | HTTP 200, `content` is `None`, `finish_reason` is `length` | The model spent `max_tokens` before answering, often on reasoning. Raise the budget and check `finish_reason` before using the text. |
| Audio content part | `audio_url` returns 400 `Failed to deserialize the JSON body into the target type` | Send audio in an `input_audio` part with base64 data and a `format` (section 5). |
| Transcript wording | Filler words and small differences, such as "costs" for "cost" | Score a transcript on keywords, not an exact match. |
| Transcript length | The transcript stops before the end of the clip, with `finish_reason` `stop` | Check that the words you need are there; a larger `max_tokens` does not help. |

## Next

- General-purpose Mistral models and the size ladder:
  [`01-mistral-text-and-sizes`](01-mistral-text-and-sizes.ipynb)
- Another coding family: [`04-qwen/02`](../04-qwen/02-qwen3-coder-and-vision.ipynb)
  (Qwen3 Coder)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)